<a href="https://colab.research.google.com/github/mdraisulislam23/All-in-one-/blob/main/Proper_raw_dataset_%20making%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
import numpy as np
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 0. FILE UPLOAD (FOR GOOGLE COLAB)
# ==========================================
file_name = 'BanglaAbuse_50K_Diverse_Synthetic.csv'

if not os.path.exists(file_name):
    print("Please select and upload 'BanglaAbuse_50K_Diverse_Synthetic.csv' from your computer:")
    uploaded = files.upload()

# ==========================================
# 1. INGEST & PROFILE
# ==========================================
print("\n--- Loading Dataset ---")
df = pd.read_csv(file_name)

print(f"Dataset Shape: {df.shape}")
print("\nTarget Label Distribution:")
print(df['profanity_label'].value_counts())

# ==========================================
# 2. DATA CLEANING & IMPUTATION
# ==========================================
print("\n--- Cleaning Data ---")

# Drop unique identifier column if present
if 'text_id' in df.columns:
    df = df.drop(columns=['text_id'])

# Remove duplicate rows
df = df.drop_duplicates()

# Fill missing values in metadata columns
categorical_cols_to_fill = ['profanity_category', 'target_type']
for col in categorical_cols_to_fill:
    if col in df.columns:
        df[col] = df[col].fillna('None')

# Handle missing text entries
df['raw_text'] = df['raw_text'].fillna('')
df['normalized_text'] = df['normalized_text'].fillna(df['raw_text'])

# Clean string spaces across string columns
string_cols = df.select_dtypes(include='object').columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

# Map target label to binary (1 = Profane, 0 = Non-profane)
df['target'] = df['profanity_label'].apply(lambda x: 1 if str(x).lower() == 'profane' else 0)

# ==========================================
# 3. FEATURE ENGINEERING
# ==========================================
print("--- Engineering Features ---")

# Compute text length and word count
df['text_char_len'] = df['normalized_text'].apply(len)
df['text_word_count'] = df['normalized_text'].apply(lambda x: len(x.split()))

# Convert binary flags (Yes/No -> 1/0)
binary_flag_cols = [
    'slang_present', 'code_mixed', 'romanized',
    'spelling_variation', 'obfuscation_present',
    'emoji_present', 'context_available'
]

for col in binary_flag_cols:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)

# ==========================================
# 4. TRAIN-TEST SPLIT
# ==========================================
print("--- Splitting Data ---")

text_feature = 'normalized_text'
meta_num_features = ['severity', 'text_char_len', 'text_word_count'] + [c for c in binary_flag_cols if c in df.columns]
meta_cat_features = ['language', 'script_type', 'region', 'district', 'dialect']

X = df[[text_feature] + meta_num_features + meta_cat_features]
y = df['target']

# Split dataset into 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train Set Size: {X_train.shape[0]} | Test Set Size: {X_test.shape[0]}")

# ==========================================
# 5. PIPELINE & TRANSFORMATIONS
# ==========================================
print("--- Building Transformers & Training Model ---")

text_transformer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
num_transformer = StandardScaler()
cat_transformer = OneHotEncoder(handle_unknown='ignore')

# Combine all transformers using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_transformer, text_feature),
        ('num', num_transformer, meta_num_features),
        ('cat', cat_transformer, meta_cat_features)
    ]
)

# Define machine learning pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Train model
pipeline.fit(X_train, y_train)

# ==========================================
# 6. EVALUATION & EXPORT
# ==========================================
print("--- Evaluating Model Performance ---")

y_pred = pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Non-profane', 'Profane']))

# Save clean dataset to Colab environment
df.to_csv('cleaned_prepared_bangla_abuse.csv', index=False)
print("\nProcessed dataset successfully saved to 'cleaned_prepared_bangla_abuse.csv'.")

Please select and upload 'BanglaAbuse_50K_Diverse_Synthetic.csv' from your computer:


Saving BanglaAbuse_50K_Diverse_Synthetic.csv to BanglaAbuse_50K_Diverse_Synthetic.csv

--- Loading Dataset ---
Dataset Shape: (50000, 21)

Target Label Distribution:
profanity_label
Profane        36090
Non-profane    13910
Name: count, dtype: int64

--- Cleaning Data ---
--- Engineering Features ---
--- Splitting Data ---
Train Set Size: 5093 | Test Set Size: 1274
--- Building Transformers & Training Model ---
--- Evaluating Model Performance ---
Accuracy: 1.0000

Classification Report:

              precision    recall  f1-score   support

 Non-profane       1.00      1.00      1.00       353
     Profane       1.00      1.00      1.00       921

    accuracy                           1.00      1274
   macro avg       1.00      1.00      1.00      1274
weighted avg       1.00      1.00      1.00      1274


Processed dataset successfully saved to 'cleaned_prepared_bangla_abuse.csv'.
